In [ ]:
# Example 1: pyproject.toml parsing and configuration management
from poetry.core.pyproject.toml import PyProjectTOML
from pathlib import Path
import tempfile
import os

# Create a sample pyproject.toml for demonstration
toml_content = """
[tool.poetry]
name = "example-project"
version = "1.2.3"
description = "Example project for poetry-core testing"
authors = ["Test Author <test@example.com>"]
license = "MIT"
readme = "README.md"

[tool.poetry.dependencies]
python = "^3.8"
requests = ">=2.25.0,<3.0.0"
click = "^8.0"

[tool.poetry.group.dev.dependencies]
pytest = "^7.0"
black = "^23.0"

[build-system]
requires = ["poetry-core>=1.0.0"]
build-backend = "poetry.core.masonry.api"
"""

# Create temporary pyproject.toml file
with tempfile.NamedTemporaryFile(mode='w', suffix='.toml', delete=False) as f:
    f.write(toml_content)
    toml_path = Path(f.name)  # Convert to Path object

try:
    # Parse the pyproject.toml using PyProjectTOML
    pyproject = PyProjectTOML(toml_path)
    
    print("=== pyproject.toml Parsing Results ===")
    print(f"Project name: {pyproject.poetry_config.get('name')}")
    print(f"Version: {pyproject.poetry_config.get('version')}")
    print(f"Description: {pyproject.poetry_config.get('description')}")
    print(f"License: {pyproject.poetry_config.get('license')}")
    
    # Extract dependencies
    deps = pyproject.poetry_config.get('dependencies', {})
    print(f"\nRuntime dependencies: {list(deps.keys())}")
    
    # Extract dev dependencies
    dev_deps = pyproject.poetry_config.get('group', {}).get('dev', {}).get('dependencies', {})
    print(f"Dev dependencies: {list(dev_deps.keys())}")
    
    # Build system configuration
    build_system = pyproject.build_system
    print(f"\nBuild backend: {build_system.build_backend}")
    print(f"Build requires: {build_system.requires}")
    
    assert pyproject.is_poetry_project()
    assert pyproject.poetry_config['name'] == "example-project"
    assert pyproject.poetry_config['version'] == "1.2.3"
    assert pyproject.poetry_config['license'] == "MIT"
    assert deps == {"python": "^3.8", "requests": ">=2.25.0,<3.0.0", "click": "^8.0"}
    assert dev_deps == {"pytest": "^7.0", "black": "^23.0"}
    assert build_system.build_backend == "poetry.core.masonry.api"
    assert build_system.requires == ["poetry-core>=1.0.0"]
    
finally:
    os.unlink(toml_path)

In [ ]:
# Example 2: Version constraint parsing and validation
from poetry.core.constraints.version import Version, VersionRange, parse_constraint

# Demonstrate various version constraint patterns used in poetry
constraint_examples = [
    ("^1.2.3", "Caret constraint - allows >=1.2.3,<2.0.0"),
    ("~1.2.3", "Tilde constraint - allows >=1.2.3,<1.3.0"),
    (">=1.0,<2.0", "Multiple constraints"),
    ("!=1.2.4", "Exclusion constraint"),
    (">=1.5.0", "Minimum version"),
]

print("=== Version Constraint Parsing ===")
matched = {}
for constraint_str, description in constraint_examples:
    constraint = parse_constraint(constraint_str)
    print(f"\nConstraint: '{constraint_str}' ({description})")
    print(f"  Parsed as: {constraint}")
    
    # Test which versions match this constraint
    test_versions = [
        Version.parse("1.0.0"),
        Version.parse("1.2.3"),
        Version.parse("1.2.4"),
        Version.parse("1.3.0"),
        Version.parse("2.0.0"),
    ]
    
    matches = [str(v) for v in test_versions if constraint.allows(v)]
    matched[constraint_str] = (str(constraint), matches)
    print(f"  Matches: {matches}")

assert matched == {
    "^1.2.3": (">=1.2.3,<2.0.0", ["1.2.3", "1.2.4", "1.3.0"]),
    "~1.2.3": (">=1.2.3,<1.3.0", ["1.2.3", "1.2.4"]),
    ">=1.0,<2.0": (">=1.0,<2.0", ["1.0.0", "1.2.3", "1.2.4", "1.3.0"]),
    "!=1.2.4": ("!=1.2.4", ["1.0.0", "1.2.3", "1.3.0", "2.0.0"]),
    ">=1.5.0": (">=1.5.0", ["2.0.0"]),
}, matched
# A caret constraint allows later minor releases but not the next major release
assert parse_constraint("^1.2.3").allows(Version.parse("1.9.0"))
assert not parse_constraint("^1.2.3").allows(Version.parse("2.0.0"))
assert isinstance(parse_constraint("^1.2.3"), VersionRange)

# Demonstrate version range operations
print("\n=== Version Range Operations ===")
range1 = VersionRange(
    min=Version.parse("1.0.0"),
    max=Version.parse("2.0.0"),
    include_min=True,
    include_max=False
)
range2 = VersionRange(
    min=Version.parse("1.5.0"),
    max=Version.parse("3.0.0"),
    include_min=True,
    include_max=False
)

print(f"Range 1: {range1}")
print(f"Range 2: {range2}")

# Calculate intersection
intersection = range1.intersect(range2)
print(f"Intersection: {intersection}")

# Test version compatibility
test_version = Version.parse("1.7.0")
print(f"\nVersion {test_version} allowed by Range 1: {range1.allows(test_version)}")
print(f"Version {test_version} allowed by Range 2: {range2.allows(test_version)}")

assert str(intersection) == ">=1.5.0,<2.0.0"
assert range1.allows(test_version) and range2.allows(test_version)
assert intersection.allows(test_version) and not intersection.allows(Version.parse("2.5.0"))
assert range1.allows(Version.parse("1.0.0")) and not range1.allows(Version.parse("2.0.0"))

In [ ]:
# Example 3: Factory - Creating Poetry projects from pyproject.toml
from poetry.core.factory import Factory
from pathlib import Path
import tempfile
import os

# Create a complete project structure
with tempfile.TemporaryDirectory() as temp_dir:
    project_dir = Path(temp_dir) / "example_project"
    project_dir.mkdir()
    
    # Create pyproject.toml
    pyproject_content = """
[tool.poetry]
name = "example-package"
version = "1.0.0"
description = "Example package created with Factory"
authors = ["Example Author <author@example.com>"]

[tool.poetry.dependencies]
python = "^3.8"
requests = "^2.28.0"
click = ">=8.0,<9.0"

[build-system]
requires = ["poetry-core>=1.0.0"]
build-backend = "poetry.core.masonry.api"
"""
    
    pyproject_path = project_dir / "pyproject.toml"
    pyproject_path.write_text(pyproject_content)
    
    # Create minimal package structure
    pkg_dir = project_dir / "example_package"
    pkg_dir.mkdir()
    (pkg_dir / "__init__.py").write_text('"""Example package."""\n__version__ = "1.0.0"\n')
    
    print("=== Creating Poetry Project with Factory ===")
    
    # Use Factory to create Poetry instance from project
    poetry = Factory().create_poetry(project_dir)
    
    print(f"Project name: {poetry.package.name}")
    print(f"Project version: {poetry.package.version}")
    print(f"Description: {poetry.package.description}")
    
    # Display dependencies
    print(f"\n=== Project Dependencies ===")
    for dep in poetry.package.all_requires:
        print(f"  - {dep.name}: {dep.constraint}")
    
    # Display package information
    print(f"\n=== Package Information ===")
    print(f"Python requirement: {poetry.package.python_constraint}")
    print(f"Authors: {poetry.package.authors}")
    
    # pyproject_path is the project file
    print(f"\n=== Project Configuration ===")
    print(f"Project root: {poetry.pyproject_path.parent}")

    assert poetry.package.name == "example-package"
    assert poetry.package.version.text == "1.0.0"
    assert poetry.package.description == "Example package created with Factory"
    assert {dep.name: str(dep.constraint) for dep in poetry.package.all_requires} == {
        "requests": ">=2.28.0,<3.0.0",
        "click": ">=8.0,<9.0",
    }
    assert str(poetry.package.python_constraint) == ">=3.8,<4.0"
    assert poetry.package.authors == ["Example Author <author@example.com>"]
    assert poetry.pyproject_path == pyproject_path

In [ ]:
# Example 4: PEP 440 version parsing and markers
from poetry.core.version.pep440 import PEP440Version
from poetry.core.version.markers import parse_marker

# Demonstrate PEP 440 version parsing and comparison
print("=== PEP 440 Version Parsing ===")
version_strings = [
    "1.0.0",
    "2.1.3rc1",
    "3.0.0a5",
    "1.5.2.post1",
    "2.0.0.dev3",
    "1.2.3+local.version",
]

versions = []
for version_str in version_strings:
    version = PEP440Version.parse(version_str)
    versions.append(version)
    print(f"  {version_str:20s} -> release {version.release.text}, pre={version.pre}, "
          f"post={version.post}, dev={version.dev}, local={version.local}")

assert [v.text for v in versions] == version_strings
assert versions[1].pre.phase == "rc" and versions[1].pre.number == 1
assert versions[2].pre.phase == "a" and versions[2].pre.number == 5
assert versions[3].post.number == 1 and versions[4].dev.number == 3
assert versions[5].local == ("local", "version")

# Sort versions according to PEP 440 ordering
print("\n=== Version Ordering (PEP 440) ===")
sorted_versions = sorted(versions)
print("Versions in sorted order:")
for v in sorted_versions:
    print(f"  {v.text}")
assert [v.text for v in sorted_versions] == [
    "1.0.0", "1.2.3+local.version", "1.5.2.post1", "2.0.0.dev3", "2.1.3rc1", "3.0.0a5",
]

# Version comparison operations
print("\n=== Version Comparisons ===")
v1 = PEP440Version.parse("2.1.0")
v2 = PEP440Version.parse("2.1.0rc1")
v3 = PEP440Version.parse("2.1.0")

print(f"{v1.text} == {v3.text}: {v1 == v3}")
print(f"{v1.text} > {v2.text}: {v1 > v2}")
print(f"{v2.text} < {v1.text}: {v2 < v1}")
assert v1 == v3 and v1 > v2 and v2 < v1
assert v2.is_prerelease() and not v1.is_prerelease()

# Environment markers, evaluated against fixed environments
print("\n=== Environment Markers ===")
linux_env = {"python_version": "3.12", "sys_platform": "linux", "extra": "dev"}
darwin_env = {"python_version": "3.12", "sys_platform": "darwin", "extra": "test"}
marker_examples = {
    'python_version >= "3.8"': (True, True),
    'sys_platform == "linux"': (True, False),
    'python_version >= "3.8" and sys_platform == "darwin"': (False, True),
    'extra == "dev"': (True, False),
}

for marker_str, expected in marker_examples.items():
    marker = parse_marker(marker_str)
    result = (marker.validate(linux_env), marker.validate(darwin_env))
    print(f"Marker: {marker_str}")
    print(f"  Parsed: {marker}")
    print(f"  Matches linux / darwin environment: {result}")
    assert str(marker) == marker_str
    assert result == expected, (marker_str, result)


In [ ]:
# Example 5: SPDX license handling and validation
from poetry.core.spdx.license import License
from poetry.core.spdx.helpers import license_by_id

# Demonstrate SPDX license lookup by ID
print("=== SPDX License Lookup ===")
license_ids = ["MIT", "Apache-2.0", "GPL-3.0", "BSD-3-Clause", "ISC"]

found = {}
for license_id in license_ids:
    spdx_license = license_by_id(license_id)
    found[license_id] = spdx_license
    print(f"\nLicense ID: {license_id}")
    print(f"  Full name: {spdx_license.name}")
    print(f"  Identifier: {spdx_license.id}")
    print(f"  OSI approved: {spdx_license.is_osi_approved}")
    print(f"  Deprecated: {spdx_license.is_deprecated}")

assert all(isinstance(lic, License) for lic in found.values())
assert {lic_id: lic.id for lic_id, lic in found.items()} == {lic_id: lic_id for lic_id in license_ids}
assert found["MIT"].name == "MIT License"
assert found["Apache-2.0"].name == "Apache License 2.0"
assert all(lic.is_osi_approved for lic in found.values())
# GPL-3.0 is a deprecated SPDX id (replaced by GPL-3.0-only / GPL-3.0-or-later)
assert found["GPL-3.0"].is_deprecated and not found["MIT"].is_deprecated

# Lookup ignores case
assert license_by_id("mit") == found["MIT"]

# An id that is not in the SPDX list gives a custom License that is not OSI approved;
# an empty id raises ValueError
print("\n=== Unknown and Empty License Ids ===")
unknown = license_by_id("NOT-A-VALID-LICENSE")
print(f"Unknown id: {unknown}")
assert unknown == License("NOT-A-VALID-LICENSE", "NOT-A-VALID-LICENSE", False, False)
try:
    license_by_id("")
except ValueError as e:
    print(f"Empty id rejected: {e}")
else:
    raise AssertionError("license_by_id('') should raise ValueError")

# Demonstrate license properties
print("\n=== License Properties ===")
mit_license = license_by_id("MIT")
print(f"License ID: {mit_license.id}")
print(f"License name: {mit_license.name}")
print(f"Is OSI approved: {mit_license.is_osi_approved}")
print(f"Is deprecated: {mit_license.is_deprecated}")
print(f"Classifier name: {mit_license.classifier_name}")
print(f"Classifier: {mit_license.classifier}")
assert mit_license.classifier_name == "MIT License"
assert mit_license.classifier == "License :: OSI Approved :: MIT License"

# Check multiple common licenses
print("\n=== Common Open Source Licenses ===")
common_licenses = [
    "MIT",
    "Apache-2.0", 
    "GPL-3.0-or-later",
    "BSD-2-Clause",
    "MPL-2.0",
]

osi_approved = []
for license_id in common_licenses:
    lic = license_by_id(license_id)
    status = "OSI" if lic.is_osi_approved else "   "
    if lic.is_osi_approved:
        osi_approved.append(lic.id)
    print(f"{status} {lic.id:20s} - {lic.name}")

print(f"\n{len(osi_approved)}/{len(common_licenses)} licenses are OSI approved")
assert osi_approved == common_licenses
